<a href="https://colab.research.google.com/github/Ekimxii/fitness_pose_tracker/blob/valerie-column-description/feedback_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## load model

In [17]:
# import libraries & models
!pip install -U sentence-transformers

import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

encoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.3/780.3 kB 8.1 MB/s  0:00:00
  Attempting uninstall: sentence-transformers
    Found existing installation: sentence-transformers 5.7.0
    Uninstalling sentence-transformers-5.7.0:
      Successfully uninstalled sentence-transformers-5.7.0


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

... (4 outputs left out)

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [18]:
# load final feedback dataset
url = "https://raw.githubusercontent.com/Ekimxii/fitness_pose_tracker/main/coach_feedback_final.csv"
df = pd.read_csv(url)
df.head()

,video_id,text,start_frame,end_frame,type,start_s,end_s,duration_s,word_count,body_part,tone,feedback_kind,vader_tone,exercise,primary_category,all_categories
0,6,First up are high knees!,397,460,transition,13.233333,15.333333,2.100000,5,none,NaN,NaN,NaN,high knees,transition,['transition']
1,6,Nice!,461,502,feedback,15.366667,16.733333,1.366667,1,none,positive,compliment,positive,high knees,transition,['transition']
2,6,Love the high knees!,503,567,feedback,16.766667,18.900000,2.133333,4,none,positive,compliment,positive,high knees,encouragement,['encouragement']
3,6,I really like the speed here. Keep it up!,574,638,feedback,19.133333,21.266667,2.133333,9,none,positive,compliment,positive,high knees,encouragement,['encouragement']
4,6,Awesome! Keep going!,664,727,feedback,22.133333,24.233333,2.100000,3,none,positive,compliment,positive,high knees,encouragement,"['encouragement', 'rep_counting', 'transition']"


## embed messages

In [19]:
# keep only actual feedback messages
feedback_df = df[df["type"]=="feedback"].copy().reset_index(drop=True)
feedback_df.head()

,video_id,text,start_frame,end_frame,type,start_s,end_s,duration_s,word_count,body_part,tone,feedback_kind,vader_tone,exercise,primary_category,all_categories
0,6,Nice!,461,502,feedback,15.366667,16.733333,1.366667,1,none,positive,compliment,positive,high knees,transition,['transition']
1,6,Love the high knees!,503,567,feedback,16.766667,18.900000,2.133333,4,none,positive,compliment,positive,high knees,encouragement,['encouragement']
2,6,I really like the speed here. Keep it up!,574,638,feedback,19.133333,21.266667,2.133333,9,none,positive,compliment,positive,high knees,encouragement,['encouragement']
3,6,Awesome! Keep going!,664,727,feedback,22.133333,24.233333,2.100000,3,none,positive,compliment,positive,high knees,encouragement,"['encouragement', 'rep_counting', 'transition']"
4,6,We're almost there with the high knees. Work i...,781,843,feedback,26.033333,28.100000,2.066667,11,none,neutral,other,neutral,high knees,encouragement,"['encouragement', 'form_correction', 'exercise..."


In [20]:
# convert feedback messages into embeddings
feedback_embeddings = encoder.encode(
    feedback_df["text"].tolist(),
    convert_to_numpy=True,
    show_progress_bar=True
)

print("Embedding shape:", feedback_embeddings.shape)

Batches:   0%|          | 0/63 [00:00<?, ?it/s]

Embedding shape: (2013, 384)


In [22]:
# Write a few fake dictionaries to simulate FORM outputs

fake_dicts = [

    # Dictionary 1: first detection of knee valgus
    {"exercise": "squat", "checkpoint": "down", "rep": 1, "t": 1.0,
        "issues": [
            {"body_part": "knee", "issue": "valgus", "status": "bad", "severity": 0.7}
              ]
    },

    # Dictionary 2: Knee valgus appears again
    {"exercise": "squat", "checkpoint": "down", "rep": 2, "t": 3.0,
        "issues": [
            {"body_part": "knee", "issue": "valgus", "status": "bad", "severity": 0.8}
              ]
    },

    # Dictionary 3: Multiple issues detected
    {"exercise": "squat", "checkpoint": "down", "rep": 3, "t": 5.0,
        "issues": [
            {"body_part": "knee", "issue": "valgus", "status": "bad", "severity": 0.8},
            {"body_part": "back", "issue": "rounding", "status": "bad", "severity": 0.9}
              ]
    },

    # Dictionary 4: Knee valgus is no longer detected
    {"exercise": "squat", "checkpoint": "down", "rep": 4, "t": 7.0,
        "issues": [
            {"body_part": "back", "issue": "rounding", "status": "bad", "severity": 0.7}
              ]
    },

    # Dictionary 5: No issues detected
    {"exercise": "squat", "checkpoint": "down", "rep": 5, "t": 9.0,
        "issues": []
    },


    # Dictionary 6: No issues detected again
    {"exercise": "squat", "checkpoint": "up", "rep": 7, "t": 13.0,
        "issues": []
    },

    # Dictionary 8: New issue detected
    {"exercise": "squat", "checkpoint": "down", "rep": 8, "t": 18.0,
        "issues": [
            {"body_part": "back", "issue": "rounding", "status": "bad", "severity": 0.85}
              ]
    }
]

# Present the fake dictionaries
for i, d in enumerate(fake_dicts):
    print(f"\nDictionary {i + 1}")
    print("Exercise:", d["exercise"])
    print("Checkpoint:", d["checkpoint"])
    print("Rep:", d["rep"])
    print("Time:", d["t"])
    print("Issues:", d["issues"])


Dictionary 1
Exercise: squat
Checkpoint: down
Rep: 1
Time: 1.0
Issues: [{'body_part': 'knee', 'issue': 'valgus', 'status': 'bad', 'severity': 0.7}]

Dictionary 2
Exercise: squat
Checkpoint: down
Rep: 2
Time: 3.0
Issues: [{'body_part': 'knee', 'issue': 'valgus', 'status': 'bad', 'severity': 0.8}]

Dictionary 3
Exercise: squat
Checkpoint: down
Rep: 3
Time: 5.0
Issues: [{'body_part': 'knee', 'issue': 'valgus', 'status': 'bad', 'severity': 0.8}, {'body_part': 'back', 'issue': 'rounding', 'status': 'bad', 'severity': 0.9}]

Dictionary 4
Exercise: squat
Checkpoint: down
Rep: 4
Time: 7.0
Issues: [{'body_part': 'back', 'issue': 'rounding', 'status': 'bad', 'severity': 0.7}]

Dictionary 5
Exercise: squat
Checkpoint: down
Rep: 5
Time: 9.0
Issues: []

Dictionary 6
Exercise: squat
Checkpoint: up
Rep: 7
Time: 13.0
Issues: []

Dictionary 7
Exercise: squat
Checkpoint: down
Rep: 8
Time: 18.0
Issues: [{'body_part': 'back', 'issue': 'rounding', 'status': 'bad', 'severity': 0.85}]


In [ ]:
# collect dictionaries in real time

## algorithm to decide

In [23]:
# algorithm to decide (based on sample algorithm, may further revise this later)

from collections import deque

class FeedbackController:
    def __init__(
        self,
        window_size=3,        # keep how many recent dicts
        min_occurrences=2,      # min number of times an issue must appear within the window to trigger feedback
        global_cooldown=4.5,     # min time between any two feedback messages
        issue_cooldown=15.0,     # min time before repeating feedback about the same issue
        recovery_frames=2,      # no. of consecutive dictionaries in which an issue must be absent to trigger recovery feedback
        praise_after_clean_frames=3, # no. of consecutive dictionaries with no detected issues before triggering general positive feedback
    ):
        # Store recent FORM dictionaries
        self.history = deque(maxlen=window_size)

        # Decision algorithm parameters
        self.min_occurrences = min_occurrences
        self.global_cooldown = global_cooldown
        self.issue_cooldown = issue_cooldown
        self.recovery_frames = recovery_frames
        self.praise_after_clean_frames = praise_after_clean_frames

        # Track feedback timing
        self.last_message_time = float("-inf")
        self.last_issue_message_time = {}

        # Track issues that have already received corrections
        self.active_issues = {}

        # Store issues that are ready for recovery encouragement
        self.pending_recoveries = set()

        # Track consecutive clean frames
        self.clean_frames = 0

        # Track current exercise
        self.current_exercise = None

    def process(self, frame_dict):
        """Process one FORM dictionary and return a feedback decision or None."""

        now = float(frame_dict["t"])
        exercise = frame_dict["exercise"]

        # Reset exercise-specific tracking when the exercise changes
        if self.current_exercise != exercise:
            self.history.clear()
            self.active_issues.clear()
            self.pending_recoveries.clear()
            self.clean_frames = 0
            self.current_exercise = exercise

        # Get all currently detected bad issues
        current_issues = {}

        for issue in frame_dict.get("issues", []):
            if issue.get("status") != "bad":
                continue

            key = (
                exercise,
                issue["body_part"],
                issue["issue"],
            )

            current_issues[key] = issue

        # Add the current dictionary to recent history
        self.history.append(frame_dict)

        # Count how many dictionaries contain each issue
        issue_counts = {}

        for item in self.history:
            frame_keys = set()

            for issue in item.get("issues", []):
                if issue.get("status") != "bad":
                    continue

                key = (
                    item["exercise"],
                    issue["body_part"],
                    issue["issue"],
                )

                frame_keys.add(key)

            # Count each issue only once per dictionary
            for key in frame_keys:
                issue_counts[key] = issue_counts.get(key, 0) + 1

        # Find persistent issues that are still present now
        persistent_issues = []

        for key, issue in current_issues.items():
            if issue_counts.get(key, 0) >= self.min_occurrences:
                persistent_issues.append((key, issue))

        # Update recovery tracking for previously corrected issues
        for key in list(self.active_issues):
            if key in current_issues:
                # The issue is still present
                self.active_issues[key]["gone_frames"] = 0
            else:
                # The issue has disappeared
                self.active_issues[key]["gone_frames"] += 1

                if (
                    self.active_issues[key]["gone_frames"]
                    >= self.recovery_frames
                ):
                    self.pending_recoveries.add(key)
                    del self.active_issues[key]

        # Update clean-frame counter
        if current_issues:
            self.clean_frames = 0
        else:
            self.clean_frames += 1

        # Rule 5: Select the most severe persistent issue
        if persistent_issues:
            key, selected_issue = max(
                persistent_issues,
                key=lambda pair: pair[1].get("severity", 0),
            )

            last_issue_time = self.last_issue_message_time.get(
                key, float("-inf")
            )

            # Check per-issue cooldown
            if now - last_issue_time >= self.issue_cooldown:

                # Check global cooldown
                if now - self.last_message_time >= self.global_cooldown:

                    self.last_issue_message_time[key] = now
                    self.last_message_time = now

                    # Track this corrected issue
                    self.active_issues[key] = {"gone_frames": 0}

                    # It is no longer waiting for recovery feedback
                    self.pending_recoveries.discard(key)

                    return {
                        "action": "constructive",
                        "exercise": key[0],
                        "body_part": key[1],
                        "issue": key[2],
                        "severity": selected_issue.get("severity", 0),
                        "query": (
                            f"{key[0]} {key[1]} {key[2]} "
                            "form correction"
                        ),
                    }

            # A persistent issue is present, but its cooldown has not expired
            return None

        # Rule 6: Praise after a previously corrected issue disappears
        if self.pending_recoveries:
            if now - self.last_message_time >= self.global_cooldown:

                key = self.pending_recoveries.pop()

                self.last_message_time = now

                return {
                    "action": "recovery",
                    "exercise": key[0],
                    "body_part": key[1],
                    "issue": key[2],
                    "query": (
                        f"{key[0]} {key[1]} "
                        f"{key[2]} improvement encouragement"
                    ),
                }

        # Rule 7: General encouragement after consecutive clean frames
        if self.clean_frames >= self.praise_after_clean_frames:
            if now - self.last_message_time >= self.global_cooldown:

                self.clean_frames = 0
                self.last_message_time = now

                return {
                    "action": "positive",
                    "exercise": exercise,
                    "query": (
                        f"{exercise} good form "
                        "positive encouragement"
                    ),
                }

        # Rule 8: Otherwise, stay silent
        return None

In [24]:
# test the algorithm with fake dicts
controller = FeedbackController()

for frame_dict in fake_dicts:
    decision = controller.process(frame_dict)
    print(frame_dict["t"], decision)

1.0 None
3.0 {'action': 'constructive', 'exercise': 'squat', 'body_part': 'knee', 'issue': 'valgus', 'severity': 0.8, 'query': 'squat knee valgus form correction'}
5.0 None
7.0 None
9.0 {'action': 'recovery', 'exercise': 'squat', 'body_part': 'knee', 'issue': 'valgus', 'query': 'squat knee valgus improvement encouragement'}
13.0 None
18.0 None


## for each dict: embed query --> cosine similairty --> top 3

In [32]:
# for each dict: embed query --> cosine similairty --> top 3

import numpy as np

def retrieve_top3(decision, feedback_df, feedback_embeddings, encoder):
    """Retrieve the 3 most relevant feedback messages based on a controller decision.
    decision: Dictionary returned by controller.process()
    feedback_df: Filtered feedback dataset
    feedback_embeddings: Precomputed embeddings of feedback_df messages
    encoder: SentenceTransformer model"""

    # If the controller decides not to give feedback --> return an empty DataFrame
    if decision is None:
        return feedback_df.iloc[0:0].copy()

    # Get the type of feedback we want to retrieve
    action = decision["action"]

    # Only retrieve actual feedback messages
    candidates = feedback_df.copy()

    # Filter by feedback category based on the action.
    if action == "constructive":
        # For constructive feedback, prioritize form corrections
        preferred = candidates[
            candidates["primary_category"].fillna("").str.lower()
            == "form_correction"
        ]

    else:
        # For recovery and positive feedback, prioritize encouragement
        preferred = candidates[
            candidates["primary_category"].fillna("").str.lower()
            == "encouragement"
        ]

        # If no encouragement messages are found, try compliments instead
        if preferred.empty:
            preferred = candidates[
                candidates["feedback_kind"].fillna("").str.lower()
                == "compliment"
            ]

    # If the preferred category has results, use it
    # Otherwise, keep all feedback candidates
    if not preferred.empty:
        candidates = preferred

    # Filter by exercise when matching messages are available.
    exercise = decision["exercise"].lower()

    exercise_match = candidates[
        candidates["exercise"].fillna("").str.lower() == exercise
    ]

    if not exercise_match.empty:
        candidates = exercise_match

    # Filter by body part when matching messages are available
    if decision.get("body_part"):
        body_part = decision["body_part"].lower()

        body_part_match = candidates[
            candidates["body_part"]
            .fillna("")
            .str.lower()
            .str.contains(body_part, regex=False)
        ]

        if not body_part_match.empty:
            candidates = body_part_match

    # Return an empty DataFrame if no candidates are available
    if candidates.empty:
        return candidates

    # Convert the current search query into an embedding
    query_embedding = encoder.encode(
        [decision["query"]],
        convert_to_numpy=True,
        normalize_embeddings=True
    )[0]

    # Get the corresponding precomputed embeddings
    # The feedback_df row order must match feedback_embeddings.
    row_positions = candidates.index.to_numpy()
    candidate_embeddings = feedback_embeddings[row_positions]

    # Normalize candidate embeddings
    # The dot product of normalized vectors equals cosine similarity
    candidate_embeddings = candidate_embeddings / np.maximum(
        np.linalg.norm(candidate_embeddings, axis=1, keepdims=True),
        1e-12
    )

    # Calculate cosine similarity between the query and candidates
    similarities = candidate_embeddings @ query_embedding

    # Select the 3 candidates with the highest similarity scores
    top_positions = np.argsort(similarities)[::-1][:3]

    top3 = candidates.iloc[top_positions].copy()
    top3["similarity"] = similarities[top_positions]

    return top3


In [33]:
# Test retrieval with fake dicts
# Create a fresh controller
controller = FeedbackController()

# Process each fake dictionary and test retrieval
for frame_dict in fake_dicts:

    decision = controller.process(frame_dict)

    print("\nTime:", frame_dict["t"])
    print("Decision:", decision)

    # Retrieve the top 3 messages if a decision is generated
    if decision is not None:
        results = retrieve_top3(
            decision,
            feedback_df,
            feedback_embeddings,
            encoder
        )

        print("\nTop 3 retrieved feedback messages:")

        if results.empty:
            print("No matching feedback messages found.")
        else:
            print(results[["text", "similarity"]].to_string(index=False))



Time: 1.0
Decision: None

Time: 3.0
Decision: {'action': 'constructive', 'exercise': 'squat', 'body_part': 'knee', 'issue': 'valgus', 'severity': 0.8, 'query': 'squat knee valgus form correction'}

Top 3 retrieved feedback messages:
                                                    text  similarity
More space between your knees and deeper squats, please.    0.444133

Time: 5.0
Decision: None

Time: 7.0
Decision: None

Time: 9.0
Decision: {'action': 'recovery', 'exercise': 'squat', 'body_part': 'knee', 'issue': 'valgus', 'query': 'squat knee valgus improvement encouragement'}

Top 3 retrieved feedback messages:
                                                                  text  similarity
                                          Nice deep squat! Keep it up!    0.510288
                   Yes! That's it! Now try to make your squats deeper!    0.480783
Good! That's a deeper squat! But your stance is too narrow, widen out!    0.475071

Time: 13.0
Decision: None

Time: 18.0
Decision

In [ ]:
# import sentence_transformers from hugging face and the generative model

In [ ]:
# write a prompt telling the model how to respond

In [ ]:
# send output to UI